#🍇 Modeling Grape Prices — From Physical Features to Market Value Objective/ブドウ価格のモデリング—外観特徴から市場価値を読み解く
  
This notebook focuses on modeling grape prices using physical characteristics and categorical attributes.

本ノートブックでは、ブドウの物理的特徴およびカテゴリ情報を用いて価格をモデル化する。

本分析の目的は単なる予測ではなく、  
**価格形成の要因を理解し、農業分野への応用可能性を示すこと**にある。



##Objective/分析目的

The objective of this modeling task is not only prediction, but **interpretation**.

We aim to answer:

- How much do physical features (size, length, width) affect price?
- Do interaction effects better represent perceived value?
- How do these compare with categorical variables such as variety group?

This reflects real-world agricultural pricing, where visual appearance and perceived quality strongly influence market value.

本モデルの目的は予測精度の向上だけではなく、**解釈性のある分析**である。

以下の問いに答えることを目指す：

- 粒径・房長・房幅といった物理特徴は価格にどの程度影響するか
- 特徴量同士の組み合わせ（interaction）は価格をより適切に説明できるか
- 品種グループなどのカテゴリ変数との比較において、どちらが重要か

本分析は、外観品質が価格に大きく影響する実際の青果市場の構造を反映している。

##　Feature Engineering Strategy/特徴量設計

We hypothesize that **price is not determined by individual features alone**, but by the **combination of visual characteristics**.

For example:
- Large grains AND long bunches → higher perceived value
- Balance between size and structure → premium appearance

本分析では、価格は単一の特徴量ではなく、  
**外観のバランス（組み合わせ）によって決まる**と仮定する。

例：
- 粒が大きく、かつ房が長い → 高級感が高い
- サイズと構造のバランス → 視覚的な価値

この仮説に基づき、以下の交互作用特徴量を導入する：

### Feature Selection
### 特徴量の選択

The model includes features related to variety system,
physical characteristics, and other relevant variables.

These features are selected based on the exploratory analysis
conducted in the previous section.

本モデルでは、品種系統、外観特性、
およびその他の関連変数を特徴量として使用します。

これらの特徴量は、前節の探索的分析の結果に基づいて選択しています。


df["grain_x_length"] = df["grain_diameter_mm"] * df["bunch_length_cm"]
df["grain_x_width"]  = df["grain_diameter_mm"] * df["bunch_width_cm"]
df["length_x_width"] = df["bunch_length_cm"] * df["bunch_width_cm"]

These features aim to capture **visual balance**, which is difficult to represent using single variables.

これにより、単一指標では表現できない「見た目の完成度」を数値化する。

## Baseline: Linear Regression (No Interaction Terms)/ベースラインモデル

We begin with a simple linear regression model using only original features.

This serves as a baseline for comparison.

まず、基本的な線形回帰モデルを構築する。

In [9]:
import pandas as pd

df = pd.read_csv("grape_data_cleaned.csv")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("grape_data_cleaned.csv")

green_list = [
    'Queen Seven', 'Haneou', 'Queen Rouge', 'Shine Muscat',
    'Fujinokagayaki', 'Seto Giants', 'Shigyoku', 'Shinku', 'Kaiji',
    'Queen Roug', 'Scarlet', 'Red Shine Muscat', 'Suiho',
    'Sunshine Red', 'Ving', 'Wasekaiji', 'Pizzutello Bianco',
    'Nouvelle Rose', 'Muscat Noir', 'Miwahime'
]
df["group"] = df["grape_variety"].apply(
    lambda x: "Shine Muscat-type" if x in green_list else "Kyoho-type"
)

df["group_binary"] = df["group"].map({"Kyoho-type": 0, "Shine Muscat-type": 1})
df_model = pd.get_dummies(
    df[[
        "grain_diameter_mm",
        "bunch_length_cm",
        "bunch_width_cm",
        "year_introduced",
        "group_binary",
        "price_yen_with_tax"
    ]],
    drop_first=True
)

X = df_model.drop("price_yen_with_tax", axis=1)
y = df_model["price_yen_with_tax"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error
print("R² score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R² score: 0.35733651871254857
MAE: 846.2561908846651


- R²: 0.357
- MAE: 846

The model shows limited explanatory power, suggesting that:
- Linear relationships alone are insufficient
- Important structural relationships are missing

説明力は限定的であり、  
単純な線形関係だけでは価格構造を十分に捉えられていない。

## Linear Regression with Interaction Terms/交互作用を含む線形回帰

Extending the model by adding interaction features to capture combined effects.

交互作用特徴量を追加し、複数の特徴量の組み合わせ効果を捉えるようモデルを拡張した。

In [11]:
df = pd.read_csv("grape_data_cleaned.csv")

green_list = [
    'Queen Seven', 'Haneou', 'Queen Rouge', 'Shine Muscat',
    'Fujinokagayaki', 'Seto Giants', 'Shigyoku', 'Shinku', 'Kaiji',
    'Queen Roug', 'Scarlet', 'Red Shine Muscat', 'Suiho',
    'Sunshine Red', 'Ving', 'Wasekaiji', 'Pizzutello Bianco',
    'Nouvelle Rose', 'Muscat Noir', 'Miwahime'
]
df["group"] = df["grape_variety"].apply(
    lambda x: "Shine Muscat-type" if x in green_list else "Kyoho-type"
)

df["group_binary"] = df["group"].map({"Kyoho-type": 0, "Shine Muscat-type": 1})

# ---- Interaction terms ----
df["grain_x_length"] = df["grain_diameter_mm"] * df["bunch_length_cm"]
df["grain_x_width"]  = df["grain_diameter_mm"] * df["bunch_width_cm"]
df["length_x_width"] = df["bunch_length_cm"] * df["bunch_width_cm"]

df["grain_x_group"] = df["grain_diameter_mm"] * df["group_binary"]
df["length_x_group"] = df["bunch_length_cm"] * df["group_binary"]
df["year_x_group"] = df["year_introduced"] * df["group_binary"]

df_model = df[[
    "grain_diameter_mm",
    "bunch_length_cm",
    "bunch_width_cm",
    "year_introduced",
    "group_binary",

    # interaction terms:
    "grain_x_length",
    "grain_x_width",
    "length_x_width",
    "grain_x_group",
    "length_x_group",
    "year_x_group",

    "price_yen_with_tax"
]]

X = df_model.drop("price_yen_with_tax", axis=1)
y = df_model["price_yen_with_tax"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R² score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


R² score: 0.4931887913444939
MAE: 723.5234909251129


- Train R²: 0.470
- Test  R²: 0.493
- Train MAE: 748
- Test  MAE: 723

Adding interaction terms significantly improves performance.

This suggests that:

 **Consumers evaluate grapes based on combined visual characteristics rather than isolated attributes**

Additional Observation:

- Train and Test scores are similar
- No strong overfitting observed

交互作用特徴量の導入により、モデル性能が大きく向上した。

消費者は単一の特徴ではなく、  
**複数の外観要素の組み合わせによって価値を判断している**

## Ridge Regression/Ridge回帰

To evaluate potential overfitting, L2 regularization is applied.

過学習の可能性を評価するため、L2正則化（Ridge回帰）を適用した。



In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

X = df_model.drop("price_yen_with_tax", axis=1)
y = df_model["price_yen_with_tax"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Ridge Regression (L2 Regularization) ----

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

print("Ridge R² score:", r2_score(y_test, y_pred_ridge))
print("Ridge MAE:", mean_absolute_error(y_test, y_pred_ridge))

Ridge R² score: 0.4646111399364067
Ridge MAE: 747.1400801175815


- R²: 0.465
- MAE: 747

Interpretation:

- Performance slightly decreases compared to standard Linear Regression
- Regularization does not provide benefit

This suggests:
- The model is not overfitting
- Feature design is more important than regularization

交互作用特徴量を導入したため、多重共線性の影響を考慮してRidge回帰を適用しました。

しかし結果として性能は改善せず、むしろわずかに低下しました。

このことから：
- モデルは過学習していない
- 正則化よりも特徴量設計の影響が大きい

と判断しました。

## Random Forest Regression/ランダムフォレスト回帰

We apply a non-linear model to test whether more complex relationships exist.

より複雑な特徴量間の関係性を検証するため、非線形モデルを用いた。

In [12]:
from sklearn.ensemble import RandomForestRegressor

X = df_model.drop("price_yen_with_tax", axis=1)
y = df_model["price_yen_with_tax"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest R² score:", r2_score(y_test, y_pred_rf))
print("Random Forest MAE:", mean_absolute_error(y_test, y_pred_rf))

importances = rf_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

importance_df

Random Forest R² score: 0.4218847299062113
Random Forest MAE: 804.2419436842117


,feature,importance
5,grain_x_length,0.441421
6,grain_x_width,0.147976
0,grain_diameter_mm,0.099285
7,length_x_width,0.076400
8,grain_x_group,0.063945
2,bunch_width_cm,0.063267
3,year_introduced,0.046392
1,bunch_length_cm,0.029294
10,year_x_group,0.016271
9,length_x_group,0.013707


- R²: 0.422
- MAE: 804

Interpretation:

- Random Forest underperforms compared to Linear Regression
- Suggests the data structure is not highly non-linear

Key takeaway:

**Well-designed features outperform model complexity**

解釈：

非線形モデルであるにもかかわらず、性能は向上しなかった。

示唆：

- データ構造は高度な非線形ではない
- **適切な特徴量設計がモデル性能を左右する**

## Feature Importance/特徴量重要度 (Random Forest)

Top features:
1. grain_x_length → 0.44  
2. grain_x_width → 0.15  
3. grain_diameter → 0.10  
4. length_x_width → 0.08  

Key Insight:

Interaction features dominate model importance.

This strongly supports the hypothesis that:

**Perceived value is driven by visual balance rather than individual dimensions**

交互作用特徴量が最も重要。

**外観の「バランス」が価格決定に大きく寄与している**

## Limitations / 制約

This analysis provides useful insights, but several limitations should be noted.

本分析は有用な示唆を提供する一方で、いくつかの制約が存在する。

Sample Size / サンプル数

- Limited sample size (~180) may affect generalizability  
- 約180件のデータのため、汎化性能に制限がある

Measurement Error / 測定誤差

- Manual measurements may introduce noise  
- 手動測定により誤差が含まれる可能性がある

Missing Variables / 重要変数の欠落

- Key factors (e.g., sugar content, brand) are not included  
- 糖度やブランドなどの重要要因が含まれていない

## Conclusion/まとめ

This study explored the factors influencing grape pricing using both physical features and engineered interaction terms.

本分析では、ブドウの物理的特徴および交互作用特徴量を用いて価格決定要因を分析した。

---

### Key Findings/主な知見

- Interaction features significantly improved model performance  
  → 交互作用特徴量の導入によりモデル性能が大きく向上した  

- Linear models outperformed more complex models  
  → 複雑なモデルよりも線形モデルの方が高い性能を示した  

- Visual balance (combined features) plays a critical role in pricing  
  → 外観の「組み合わせ（バランス）」が価格に大きく影響する  

### Interpretation/解釈

These results suggest that grape pricing is driven more by structured feature relationships than by highly complex non-linear patterns.

本結果は、価格が高度な非線形性よりも、特徴量同士の構造的な関係によって説明されることを示唆している。

### Future Perspective/今後の展開

To further enhance this analysis:

本分析の発展として、以下が考えられる：

- Expanding the dataset across regions and seasons  
  → 地域・季節データの拡張  

- Incorporating additional quality indicators (e.g., sweetness, branding)  
  → 糖度やブランドなどの品質指標の追加  

- Applying the framework to export markets (e.g., Hong Kong)  
  → 海外市場（香港など）への応用  

### Final Insight/最終的な示唆

Feature engineering plays a more critical role than model complexity in explaining grape pricing.

ブドウ価格の説明においては、モデルの複雑さよりも特徴量設計の方が重要である。